# 05 — Ensemble Learning

Notebook 05 reuses the validation prediction CSVs and provenance saved by **Notebook 04 — Model Evaluation**. It implements majority voting, equal/weighted soft voting, validation-only weight optimization and ensemble freezing, then offers a separate opt-in to evaluate every individual model and voting strategy on the same untouched test split.

All execution flags default to False. The optimization step does not load CNNs, decode images or regenerate validation predictions. Actual test inference requires the same private checkpoints and frozen manifest/context used by Notebook 04. No weights/datasets are downloaded, no models are retrained and no experimental results are prefilled.

## 1. Shared run settings
Use the same Phase 3 session, original manifest, checkpoint/image-root mappings and evaluation name as Notebook 04. Do not change frozen splits, preprocessing or architectures. The output root may point to a fresh writable copy of the **pre-ensemble** Notebook 04 handoff; that copy changes no prediction or provenance bytes.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import copy
import json

RUN_PLATFORM_SETUP = False
CLONE_REPOSITORY = False
INSTALL_DEPENDENCIES = False
RUN_ENSEMBLE_LEARNING = False
RUN_FINAL_TEST = False
SHOW_SAVED_TABLES = False

PLATFORM = "kaggle" if (os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").is_dir()) else "colab" if Path("/content").is_dir() else "local"
default_repo = "/kaggle/working/glaucoma_detection" if PLATFORM == "kaggle" else "/content/glaucoma_detection" if PLATFORM == "colab" else str(Path.cwd() if (Path.cwd()/"src").is_dir() else Path.cwd().parent)
REPO_ROOT = Path(os.environ.get("PROJECT_REPO", default_repo))
REPO_URL = "https://github.com/VDJ2824/glaucoma_detection.git"
REPO_REF = "main"  # prefer a reviewed commit for reproducibility
BUNDLE_ROOT = Path("/kaggle/input/your-private-training-run")  # EDIT: run containing checkpoints/manifests/environment
if PLATFORM == "kaggle":
    default_session = BUNDLE_ROOT / "environment/session.json"
    default_output = Path("/kaggle/working/artifacts")
elif PLATFORM == "colab":
    default_output = Path("/content/drive/MyDrive/glaucoma_detection/artifacts")
    default_session = default_output / "runs/development/environment/session.json"
else:
    default_output = REPO_ROOT / "artifacts"
    default_session = default_output / "runs/development/environment/session.json"
SESSION_FILE = os.environ.get("GLAUCOMA_SESSION", str(default_session))
MANIFEST_CSV = BUNDLE_ROOT / "manifests/acrima_metadata.csv" if PLATFORM == "kaggle" else None
CHECKPOINT_ROOT = BUNDLE_ROOT / "checkpoints/core" if PLATFORM == "kaggle" else None
# Optional per-model bases containing <model>/<stage>/; leave None for a common base/session setting.
CHECKPOINT_ROOTS = None
# To decode original Kaggle manifest paths on another machine, set the CURRENT flat image directory.
# Keep the original CSV unchanged; this affects only in-memory physical path lookup.
FLAT_IMAGE_ROOT = None
OUTPUT_ROOT = default_output
EVALUATION_NAME = "acrima_phase4_v1"

COPY_VALIDATION_HANDOFF = False
VALIDATION_HANDOFF_SOURCE = None  # root of saved Notebook 04 phase4/<name> run
if RUN_ENSEMBLE_LEARNING and RUN_FINAL_TEST:
    raise ValueError("Run ensemble learning/freeze alone, review it, then disable it before final test opt-in.")


## 2. Independent platform setup
These are path/bootstrap settings only; all data/prediction/metric/selection logic is reused from project modules. Colab mounts Drive explicitly. Kaggle output folders are ephemeral: attach Notebook 04's saved output and stage its verified handoff into writable storage when using a separate notebook runtime. Installing packages is optional and requires restart; no validation inference runs in this step.

In [ ]:
if RUN_PLATFORM_SETUP:
    if not (3, 10) <= sys.version_info[:2] < (3, 14):
        raise RuntimeError("Use Python 3.10–3.13.")
    if PLATFORM == "colab":
        from google.colab import drive
        drive.mount("/content/drive")
    if CLONE_REPOSITORY:
        if REPO_ROOT.exists():
            raise FileExistsError("Existing clone preserved; inspect it rather than resetting it.")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_REF], check=True)
    if INSTALL_DEPENDENCIES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT/"requirements-training.txt")], cwd=REPO_ROOT, check=True)
        raise RuntimeError("Restart the kernel, disable installation, then rerun.")
if not (REPO_ROOT/"src/glaucoma").is_dir():
    raise FileNotFoundError("Set REPO_ROOT to the existing project clone.")
sys.path.insert(0, str(REPO_ROOT/"src"))


## 3. Load saved session and predefine the ensemble procedure
The fixed default is a 0.05-step simplex grid plus exact equal weights (232 candidates), nonnegative weights summing to one, minimizing validation BCE clipped at 1e-7. Loss ties within 1e-12 prefer closest-to-equal weights then lexicographic member order. The threshold remains the 0.5 predefined by Notebook 04; it is not tuned here. Boundary weights may select a single model. No ensemble is assumed to beat individuals.

In [ ]:
from glaucoma.utils.notebook_runtime import load_session
from glaucoma.ensemble.predictions import MEMBERS
from glaucoma.ensemble.weights import search_protocol
from glaucoma.evaluation.workflow import (run_directory, copy_model_validation, load_saved_validation,
    run_ensemble_validation, load_frozen, run_final_test)

cfg = copy.deepcopy(load_session(SESSION_FILE))
manifest_path = Path(MANIFEST_CSV or cfg["metadata"]["output"])
checkpoint_roots = CHECKPOINT_ROOTS or ({name: str(CHECKPOINT_ROOT) for name in MEMBERS} if CHECKPOINT_ROOT is not None else None)
selection_records = {
    "mobilenet_v2": REPO_ROOT/"docs/experiments/mobilenet_v2_kaggle_2026-11-26/metrics/selection.json",
    "efficientnet_b0": REPO_ROOT/"docs/experiments/efficientnet_b0_kaggle_2026-11-27/metrics/selection.json",
    "resnet50": REPO_ROOT/"docs/experiments/resnet50_kaggle_2026-11-27/metrics/selection.json",
}
directory = run_directory(cfg, EVALUATION_NAME, OUTPUT_ROOT)
print("Manifest:", manifest_path)
print("Evaluation artifacts:", directory)
print("Member order:", MEMBERS)

PROTOCOL = search_protocol(grid_denominator=20, threshold=0.5)
print(json.dumps(PROTOCOL, indent=2))


## 4. Optional Kaggle handoff staging (no inference)
If Notebook 04 outputs are attached read-only, set `VALIDATION_HANDOFF_SOURCE` to its saved `phase4/<name>` directory and opt into copying. The helper verifies cached prediction hashes and copies only the individual-validation report tree into a fresh writable destination. It refuses frozen/test-consumed or already-search-started sources and never overwrites an existing run. Source reports and original data remain unchanged. Leave copying disabled when using the same persistent Drive folder or working directory.

In [ ]:
if COPY_VALIDATION_HANDOFF:
    if VALIDATION_HANDOFF_SOURCE is None:
        raise ValueError("Set VALIDATION_HANDOFF_SOURCE to the saved Notebook 04 evaluation-run root.")
    print("Staged verified model-validation handoff:", copy_model_validation(VALIDATION_HANDOFF_SOURCE, directory))
else:
    print("Handoff copy disabled; reusing the configured run directory.")


## 5. Inspect cached validation scores
Optional inspection verifies the Notebook 04 record checksum, source CSV hashes, original manifest hash, physical/preprocessing context, image-key alignment and ground truth. CSVs use round-trip float parsing and string image IDs. No CNN/checkpoint loading or validation image decoding occurs. Test labels and predictions are not used by this inspection.

In [ ]:
if SHOW_SAVED_TABLES:
    cached_validation, validation_record = load_saved_validation(cfg, directory, manifest_path, FLAT_IMAGE_ROOT)
    print(cached_validation.head().to_string(index=False))
    print("Cached validation rows:", len(cached_validation))
else:
    print("Cached validation inspection disabled.")


## 6. Validation-only voting, optimization and freeze
Enable only `RUN_ENSEMBLE_LEARNING`. The shared function requires a completed Notebook 04 handoff, records the predefined search, reuses its exact cached predictions and compares majority, equal soft voting and weighted soft voting. Candidate weights/losses, comparisons, confusion matrices, ROC plots and disagreement analysis are saved without overwriting the individual reports.

The immutable `frozen_ensemble.json` binds selected weights, the unchanged threshold, protocol, manifest/checkpoint/training-contract/graph hashes and the Notebook 04 handoff. Any modified source CSV/provenance is rejected. Do not rerun Notebook 04 or alter parameters after this point. Ensemble validation metrics are selection-biased and are not test performance.

In [ ]:
ensemble_result = None
if RUN_ENSEMBLE_LEARNING:
    ensemble_result = run_ensemble_validation(cfg, manifest_path, enabled=True,
        name=EVALUATION_NAME, output_root=OUTPUT_ROOT, flat_image_root=FLAT_IMAGE_ROOT,
        protocol=PROTOCOL)
    print(json.dumps(ensemble_result, indent=2))
else:
    print("Ensemble learning/freezing disabled. No weights selected.")


## 7. Review the frozen configuration
Review the saved configuration and validation tables before deliberately opting into the test step. Disable ensemble learning before setting the final-test flag; both cannot run in one notebook execution. The threshold/weights cannot be changed using test outcomes, and test access is blocked until the frozen configuration and cached source provenance are valid.

In [ ]:
if SHOW_SAVED_TABLES and (directory/"frozen_ensemble.json").is_file():
    import pandas as pd
    print(json.dumps(load_frozen(directory), indent=2))
    print(pd.read_csv(directory/"validation/ensemble/metrics_comparison.csv").to_string(index=False))


## 8. Final untouched-test evaluation — separate opt-in
Only after freezing, set `RUN_FINAL_TEST=True` and `RUN_ENSEMBLE_LEARNING=False`. The shared final-test function rechecks every fingerprint and records test access before decoding test images. It evaluates the three individuals, hard majority, equal soft voting and frozen weighted soft voting on the same aligned test rows and threshold, using unchanged preprocessing without augmentation.

Metrics are accuracy, precision, recall, specificity, F1, score-based ROC-AUC/BCE, confusion matrices, ROC curves and disagreement analysis. Hard majority has no probability-based AUC/BCE; undefined values are null with reasons. A one-attempt test marker prevents repeated test-driven retuning; investigate failures without deleting evidence or modifying the frozen protocol.

In [ ]:
test_result = None
if RUN_FINAL_TEST:
    test_result = run_final_test(cfg, manifest_path, enabled=True,
        name=EVALUATION_NAME, output_root=OUTPUT_ROOT, expected_records=selection_records,
        checkpoint_roots=checkpoint_roots, flat_image_root=FLAT_IMAGE_ROOT)
    print(json.dumps(test_result, indent=2))
else:
    print("Final test evaluation disabled. No test images accessed.")


## Artifacts and interpretation
Notebook 04 individual reports remain unchanged under `validation/`. Notebook 05 comparisons go under `validation/ensemble/`; candidate search remains `validation/weight_search.csv`. Fixed-protocol final results go under `test/`. Save Kaggle working outputs/export them privately or use mounted Drive. Do not commit predictions, datasets, model weights or ZIPs.

Image-level splitting does not prove patient independence, single-dataset evaluation does not establish external generalization, and validation optimization need not improve test performance. No real Phase 4 scores are supplied with this implementation. See the [Phase 4 guide](../docs/phase4-ensemble-evaluation.md).